# REIT Valuation Engine — Exploratory Data Analysis
End-to-end walkthrough: raw data profiling, cleaning/outlier rejection, feature engineering, micro-market clustering, valuation modeling, REIT metrics, and explainability — using the same `src/` modules the production pipeline runs.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Raw data profile

In [ ]:
from src.data_pipeline import load_raw_data

raw_df = load_raw_data()
print(raw_df.shape)
raw_df.describe(include="all").T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(raw_df["price"], bins=100, ax=axes[0], log_scale=(True, False))
axes[0].set_title("Price distribution (log-x)")
sns.histplot(raw_df["area"], bins=100, ax=axes[1])
axes[1].set_title("Area distribution")
sns.histplot(raw_df["price_per_sqft"], bins=100, ax=axes[2])
axes[2].set_title("Price/sqft distribution")
plt.tight_layout()

In [ ]:
print("Rows with lat/lon outside plausible MMR bounding box:")
from src.config import MMR_LAT_BOUNDS, MMR_LON_BOUNDS
bad_geo = raw_df[~raw_df["latitude"].between(*MMR_LAT_BOUNDS) | ~raw_df["longitude"].between(*MMR_LON_BOUNDS)]
print(len(bad_geo), "rows")
bad_geo[["locality", "latitude", "longitude"]].head(10)

## 2. Cleaning + outlier rejection

In [ ]:
from src.data_pipeline import apply_domain_bounds, reject_outliers_iqr, reject_outliers_zscore, impute_missing

df = apply_domain_bounds(raw_df)
df = reject_outliers_iqr(df)
df = reject_outliers_zscore(df)
df = impute_missing(df)
print(f"Retained {len(df):,} / {len(raw_df):,} rows ({len(df) / len(raw_df):.1%})")

## 3. Feature engineering

In [ ]:
from src.data_pipeline import engineer_features

df, feature_artifacts, locality_encoder = engineer_features(df, fit_encoders=True)
df[["macro_zone", "age_bin", "amenity_score", "locality_target_enc", "locality_freq_enc"]].sample(10, random_state=1)

In [ ]:
df["macro_zone"].value_counts().plot(kind="bar", figsize=(7, 4), title="Properties per macro zone")
plt.tight_layout()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="macro_zone", y="price_per_sqft")
plt.xticks(rotation=30, ha="right")
plt.title("Price/sqft by macro zone")
plt.tight_layout()

## 4. Micro-market clustering (PCA + K-Means)

In [ ]:
from src.clustering import run_clustering_pipeline

clustered_df, clustering_result = run_clustering_pipeline(df, save=True)
print(f"Best k = {clustering_result.best_k}, silhouette = {clustering_result.silhouette:.3f}, "
      f"Davies-Bouldin = {clustering_result.davies_bouldin:.3f}")
clustering_result.cluster_profile

In [ ]:
plt.figure(figsize=(7, 6))
sample = clustered_df.sample(min(8000, len(clustered_df)), random_state=42)
sns.scatterplot(data=sample, x="pca_1", y="pca_2", hue="market_tier", alpha=0.5, s=15)
plt.title("PCA projection colored by market tier")
plt.tight_layout()

## 5. Valuation model (LightGBM vs XGBoost, Optuna-tuned)

In [ ]:
from src.valuation_model import train_valuation_models

# Use a small trial budget for a fast notebook run; the production CLI
# (`python -m src.valuation_model`) uses the full OPTUNA_TRIALS budget.
final_model, evaluations = train_valuation_models(clustered_df, n_trials=10, save=True)
for name, ev in evaluations.items():
    print(f"{name}: RMSE={ev.rmse:,.0f}  MAE={ev.mae:,.0f}  R2={ev.r2:.4f}  MAPE={ev.mape:.2%}")

## 6. REIT metrics

In [ ]:
from src.reit_metrics import compute_property_reit_metrics, simulate_portfolio_returns

scored = compute_property_reit_metrics(clustered_df, value_column="price")
scored.groupby("market_tier")[["cap_rate", "price_to_rent_ratio", "estimated_monthly_rent"]].mean()

In [ ]:
sim_result = simulate_portfolio_returns(scored.sample(2000, random_state=1))
print(f"Expected annualized return: {sim_result.expected_annualized_return:.2%}")
print(f"Volatility: {sim_result.return_volatility:.2%}")
print(f"Sharpe ratio: {sim_result.sharpe_ratio:.2f}")
print(f"5% VaR (total return): {sim_result.value_at_risk_95:.2%}")

## 7. Explainability (SHAP)

In [ ]:
from src.explainability import build_shap_explainer, compute_shap_values, plot_shap_summary, plot_shap_waterfall
from src.valuation_model import TRAINING_FEATURE_COLUMNS

X_sample = clustered_df[TRAINING_FEATURE_COLUMNS].sample(1000, random_state=1)
explainer = build_shap_explainer(final_model)
shap_values = compute_shap_values(explainer, X_sample)
plot_shap_summary(shap_values, X_sample)
plt.show()

In [ ]:
single_row = clustered_df[TRAINING_FEATURE_COLUMNS].sample(1, random_state=7)
plot_shap_waterfall(explainer, single_row)
plt.show()